# Kontur Analizi, Geometrik Ölçüm ve Şekil Tanıma

Bu modül; ikili görüntüler üzerinden nesne sınırlarının çıkarılmasını sağlayan **kontur analizi** (`cv2.findContours`), moment tabanlı alan/çevre hesaplamaları ve çokgen yaklaşımı (**Douglas-Peucker Algoritması**) ile geometrik şekillerin sınıflandırılmasını kapsar.

---

## 1. Temel Kavramlar

- **Kontur (Contour):** Aynı renk veya yoğunluk değerine sahip sınır noktalarını bağlayan kapalı eğridir.
- **Alan (Area):** Green teoremi yardımıyla hesaplanan piksel yüzölçümü:
  $$A = \frac{1}{2} \left| \sum_{i=0}^{n-1} (x_i y_{i+1} - x_{i+1} y_i) \right|$$
- **Çevre Uzunluğu (Arc Length):** Konturu oluşturan komşu noktalar arasındaki Öklid mesafelerinin toplamı.
- **Ağırlık Merkezi (Centroid):** Görüntü momentleri kullanılarak hesaplanan geometrik merkez:
  $$C_x = \frac{M_{10}}{M_{00}}, \quad C_y = \frac{M_{01}}{M_{00}}$$
- **Douglas-Peucker Çokgen Yaklaşımı:** Bir eğrinin köşe sayısını, eğriye en uzak noktaları tolerans parametresi $\epsilon$ ile test ederek basitleştiren algoritmadır:
  $$\epsilon = \text{oran} \times \text{arcLength}$$


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Sentetik Geometrik Şekiller Görüntüsünü Yükleme
image = cv2.imread('geometric_shapes.png')
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# İkili (Binary) Görüntüye Dönüştürme
_, thresh = cv2.threshold(gray, 240, 255, cv2.THRESH_BINARY_INV)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.title("Orijinal Renkli Şekiller")
plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("İkili Eşiklenmiş Maske (Binary)")
plt.imshow(thresh, cmap='gray')
plt.axis('off')
plt.show()


## 2. Konturların Tespiti ve Geometrik Öznitelik Çıkarımı

In [ ]:
# cv2.findContours ile dış konturları bulma
contours, hierarchy = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Toplam Tespit Edilen Kontur Sayısı: {len(contours)}")

output_image = image.copy()

for i, cnt in enumerate(contours):
    area = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, closed=True)
    
    # Küçük gürültüleri filtreleme
    if area < 100:
        continue
        
    # Momentler ile merkez noktası
    M = cv2.moments(cnt)
    if M["m00"] != 0:
        cX = int(M["m10"] / M["m00"])
        cY = int(M["m01"] / M["m00"])
    else:
        cX, cY = 0, 0
        
    # Douglas-Peucker Çokgen Yaklaşımı
    epsilon = 0.03 * perimeter
    approx = cv2.approxPolyDP(cnt, epsilon, closed=True)
    num_vertices = len(approx)
    
    # Şekil Sınıflandırma Mantığı
    shape_name = "Bilinmeyen"
    if num_vertices == 3:
        shape_name = "Ucgen"
    elif num_vertices == 4:
        # En-boy oranı kontrolü (Kare vs Dikdörtgen)
        x, y, w, h = cv2.boundingRect(approx)
        aspect_ratio = float(w) / h
        shape_name = "Kare" if 0.95 <= aspect_ratio <= 1.05 else "Dikdortgen"
    elif num_vertices == 5:
        shape_name = "Besgen"
    else:
        shape_name = "Daire"
        
    print(f"Kontur {i+1}: Kose={num_vertices}, Alan={area:.1f}, Cevre={perimeter:.1f} -> {shape_name}")
    
    # Görsel üzerine kontur ve etiket çizimi
    cv2.drawContours(output_image, [approx], -1, (0, 0, 0), 3)
    cv2.circle(output_image, (cX, cY), 5, (255, 0, 0), -1)
    cv2.putText(output_image, shape_name, (cX - 35, cY - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 0), 2)

plt.figure(figsize=(10, 8))
plt.title("Sınıflandırılmış Geometrik Şekiller ve Kontur Hatları")
plt.imshow(cv2.cvtColor(output_image, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()


## 3. Sınırlayıcı Dikdörtgenler ve Dış Çemberler

In [ ]:
bounding_demo = image.copy()

for cnt in contours:
    if cv2.contourArea(cnt) < 100:
        continue
    # 1. Düz Sınırlayıcı Kutu (Bounding Box)
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(bounding_demo, (x, y), (x + w, y + h), (0, 255, 0), 2)
    
    # 2. Minimum Alanlı Döndürülmüş Dikdörtgen
    rect = cv2.minAreaRect(cnt)
    box = cv2.boxPoints(rect)
    box = np.int32(box)
    cv2.drawContours(bounding_demo, [box], 0, (255, 0, 0), 2)
    
    # 3. Minimum Sınırlayıcı Çember
    (cx, cy), radius = cv2.minEnclosingCircle(cnt)
    cv2.circle(bounding_demo, (int(cx), int(cy)), int(radius), (0, 0, 255), 2)

plt.figure(figsize=(10, 8))
plt.title("Yesil: Bounding Rect | Mavi: MinAreaRect | Kirmizi: MinEnclosingCircle")
plt.imshow(cv2.cvtColor(bounding_demo, cv2.COLOR_BGR2RGB))
plt.axis('off')
plt.show()
